In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from thefuzz import process, fuzz
import re

def obtener_indice_imsdb():
    """Obtiene el índice completo de películas de IMSDb con sus URLs de scripts."""
    url = "https://imsdb.com/all-scripts.html"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    try:
        res = requests.get(url, headers=headers, timeout=20)
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, 'html.parser')
        
        dict_urls = {}
        
        # Obtener todos los links
        all_links = soup.find_all('a', href=True)
        print(f"Total de links encontrados: {len(all_links)}")
        
        # Extraer solo los títulos de películas (que tienen /Movie Scripts/ en href)
        script_count = 0
        for link in all_links:
            href = link.get('href', '')
            titulo_raw = link.get_text(strip=True)
            
            # Filtrar solo links que apunten a scripts de películas
            if '/Movie Scripts/' in href and '.html' in href.lower():
                # Limpiar el título: remover ratings y espacios extra
                titulo = re.sub(r'\s*\d+/10\s*', '', titulo_raw).strip()
                
                # Validaciones
                if not titulo or len(titulo) < 2 or len(titulo) > 300:
                    continue
                
                # Construir URL con el título formateado
                # El formato es: /scripts/TITULO-FORMATEADO.html
                titulo_url = titulo.replace(' ', '-').replace("'", '')
                # Remover caracteres especiales
                titulo_url = re.sub(r'[^a-zA-Z0-9\-]', '', titulo_url)
                # Remover guiones múltiples
                titulo_url = re.sub(r'-+', '-', titulo_url)
                
                url_final = f"https://imsdb.com/scripts/{titulo_url}.html"
                
                # Agregar al diccionario (evitar duplicados)
                if titulo not in dict_urls:
                    dict_urls[titulo] = url_final
                    script_count += 1
        
        print(f"✅ Se extrajeron {script_count} títulos de películas")
        print(f"✅ Se obtuvieron {len(dict_urls)} películas únicas de IMSDb\n")
        
        return dict_urls
        
    except Exception as e:
        print(f"❌ Error al obtener índice: {e}")
        return {}

In [27]:
# DEBUG: Verificar el índice obtenido
print("="*70)
print("DIAGNÓSTICO DEL ÍNDICE")
print("="*70)

# Obtener el índice
indice_online = obtener_indice_imsdb()

print(f"\nTotal de películas en el índice: {len(indice_online)}")

if len(indice_online) > 0:
    print("\nPrimeras 10 películas y sus URLs:")
    print("-"*70)
    for i, (titulo, url) in enumerate(list(indice_online.items())[:10], 1):
        print(f"{i:2d}. {titulo[:50]}")
        print(f"    URL: {url}")
    
    print("\n" + "="*70)
    print("PRUEBA DE DESCARGA")
    print("="*70)
    
    # Probar con la primera película
    primer_titulo, primer_url = list(indice_online.items())[0]
    print(f"\nProbando descarga de: {primer_titulo}")
    print(f"URL: {primer_url}\n")
    
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        res = requests.get(primer_url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        res.raise_for_status()
        
        print(f"✅ Status code: {res.status_code}")
        print(f"✅ Content length: {len(res.text)} caracteres")
        
        soup = BeautifulSoup(res.text, 'html.parser')
        pre = soup.find('pre')
        
        if pre:
            texto = pre.get_text()
            tamaño = len(texto.strip())
            print(f"✅ Encontrado <pre> con {tamaño} caracteres")
            print(f"\nPrimeros 300 caracteres:\n{'-'*70}")
            print(texto[:300])
            print(f"{'-'*70}")
        else:
            print("❌ No se encontró <pre>")
            print("\nEtiquetas principales en la página:")
            for tag_name in ['pre', 'div', 'p', 'table', 'body']:
                tags = soup.find_all(tag_name)
                if tags:
                    sizes = [len(tag.get_text()) for tag in tags]
                    print(f"  {tag_name}: {len(tags)} encontrados (tamaños: {sizes[:3]}...)")
                
    except requests.exceptions.Timeout:
        print("❌ Timeout al descargar")
    except requests.exceptions.HTTPError as e:
        print(f"❌ Error HTTP: {e}")
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print("❌ No se obtuvieron películas del índice")

DIAGNÓSTICO DEL ÍNDICE
Total de links encontrados: 1364
✅ Se extrajeron 1298 títulos de películas
✅ Se obtuvieron 1298 películas únicas de IMSDb


Total de películas en el índice: 1298

Primeras 10 películas y sus URLs:
----------------------------------------------------------------------
 1. PredatorMaster and CommanderWhite ChristmasFantast
    URL: https://imsdb.com/scripts/PredatorMaster-and-CommanderWhite-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
 2. Master and CommanderWhite ChristmasFantastic Beast
    URL: https://imsdb.com/scripts/Master-and-CommanderWhite-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
 3. White ChristmasFantastic Beasts: The Crimes of Gri
    URL: https://imsdb.com/scripts/White-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
 4. Fantastic Beasts: The Crimes of GrindelwaldLegend
    URL: https://imsdb.com/scripts/Fantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
 5. Legend
    URL: https://imsdb.com/scr

In [ ]:
import pandas as pd

# Crear un DataFrame con los títulos y links
df_links = pd.DataFrame(list(indice_online.items()), columns=['Título', 'URL'])

# Ignorar los primeros 5 registros
df_links_filtered = df_links.iloc[5:].reset_index(drop=True)

# Mostrar todos los links
print(f"Total de películas (sin los primeros 5): {len(df_links_filtered)}\n")
print(df_links_filtered.to_string())

# Opcionalmente, guardar a CSV para revisar
df_links_filtered.to_csv('links_peliculas.csv', index=False)
print(f"\n✓ Se guardó 'links_peliculas.csv' con todos los links")

Total de películas: 1298

                                                                                            Título                                                                                                                         URL
0     PredatorMaster and CommanderWhite ChristmasFantastic Beasts: The Crimes of GrindelwaldLegend  https://imsdb.com/scripts/PredatorMaster-and-CommanderWhite-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
1             Master and CommanderWhite ChristmasFantastic Beasts: The Crimes of GrindelwaldLegend          https://imsdb.com/scripts/Master-and-CommanderWhite-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
2                                 White ChristmasFantastic Beasts: The Crimes of GrindelwaldLegend                              https://imsdb.com/scripts/White-ChristmasFantastic-Beasts-The-Crimes-of-GrindelwaldLegend.html
3                                                Fantastic Beasts: The Crimes of G